# 06 May 05 Default Best Fixed Baseline (Apple MPS)

Same experiment as `01_default_best_fixed_baseline.ipynb`, but DQN and RewardNet train on **Metal (MPS)** when PyTorch reports it as available.

Use **`SHOW_PROGRESS`** to enable tqdm during training and the Optuna progress bar without turning on full **`VERBOSE`** logging.

Requires macOS with Apple Silicon and a PyTorch build with `torch.backends.mps`. On Linux/Windows or without MPS, set `TORCH_DEVICE = "cpu"` or `"auto"`.

In [4]:
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import torch

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess
from simulator.model.drlb.torch_device import resolve_training_device

_mps = getattr(torch.backends, 'mps', None)
print('torch:', torch.__version__, '| mps available:', bool(_mps and _mps.is_available()))

torch: 2.11.0 | mps available: True


In [5]:
# One of: "mps" | "cpu" | "cuda" | "auto"
TORCH_DEVICE = 'mps'

RUN_NAME = 'may05_default_best_fixed_baseline_mps'
DRLB_PROFILE = 'may05_default_best_fixed'
VERBOSE = False
SHOW_PROGRESS = True  # tqdm during fit / Optuna bar (independent of VERBOSE)

resolve_training_device(TORCH_DEVICE)

device(type='mps')

In [6]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
config = replace(config, n_trials=1, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')
base_drlb_params['torch_device'] = TORCH_DEVICE

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    show_progress=SHOW_PROGRESS,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'torch_device': TORCH_DEVICE,
    'show_progress': SHOW_PROGRESS,
    'base_drlb_params': base_drlb_params,
    'reference_model_params': reference_model_params,
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}

autobidder_check campaigns: 100%|██████████| 257/257 [00:41<00:00,  6.15campaign/s, campaign_id=7.46e+7]
[I 2026-05-05 12:02:54,920] A new study created in memory with name: no-name-db1f9fc7-bc53-4288-838c-0b3f81b23e2e
Best trial: 0. Best value: 2298.77: 100%|██████████| 1/1 [15:22<00:00, 922.21s/it]


[I 2026-05-05 12:18:17,133] Trial 0 finished with value: 2298.772077196292 and parameters: {'dqn_lr': 0.0003, 'reward_net_lr': 0.01, 'bid_lower_clip': 3, 'bid_upper_clip': 8}. Best is trial 0 with value: 2298.772077196292.


autobidder_check campaigns: 100%|██████████| 257/257 [00:36<00:00,  7.14campaign/s, campaign_id=7.46e+7]


{'run_name': 'may05_default_best_fixed_baseline_mps',
 'profile': 'may05_default_best_fixed',
 'torch_device': 'mps',
 'show_progress': True,
 'base_drlb_params': {'max_bid': 100.0,
  'T': 72,
  'lambda_min': -inf,
  'lambda_max': inf,
  'bids_per_timestep': 1,
  'dqn_soft_update_tau': 0.01,
  'dqn_loss_type': 'smooth_l1',
  'dqn_grad_clip_norm': 5.0,
  'dqn_reward_clip_value': 10.0,
  'init_lambda': 0.0028423174374845716,
  'init_lambda_mode': 'constant',
  'bid_lower_clip': 3,
  'bid_upper_clip': 8,
  'traffic_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/traffic_share.csv',
  'torch_device': 'mps'},
 'reference_model_params': {'dqn_gamma': 1.0,
  'dqn_lr': 0.0003,
  'dqn_target_update_interval': 100,
  'reward_net_lr': 0.01,
  'dqn_epsilon_start': 0.95,
  'dqn_epsilon_end': 0.05,
  'dqn_epsilon_anneal': 2e-05},
 'best_val_metrics': {'cpc_relative': 1401.4261768002036,
  'rmse': 1.199764167129981,
  'clicks_sum': 2298.772077196292,
  'quickspend': 0.015564202

In [7]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])

,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
